## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **e1d80b99-32b3-4f80-9185-e5560873f257**


## Dataset

# Potentiel photovoltaïque &#40;PV&#41; par bâtiment


## Description

Produit secondaire. La couche du potentiel photovoltaïque par bâtiment &#40;EGID&#41; représente et regroupe les informations relatives au potentiel d&#x27;énergie PV. Il est basé sur le potentiel solaire dont l&#x27;irradiation est au minimum de 800 kWh/m2/an sur les toitures, tenant compte des ombrages, de la pente et de l&#x27;orientation &#40;cf. données relatives à la répartition surfacique de ce potentiel dans la couche OCEN&#95;SOLAIRE&#95;BASE&#95;BATIMENT&#41;. Le seuil de 800 kWh/m2/an est choisi de sorte une garantir une production et rentabilité économique minimale. Cela permet d&#x27;éliminer les zones d&#x27;ombrages majeures, tout en laissant à disposition une grande partie des toitures, y compris celles orientées nord mais faiblement inclinées. La couche fournit les informations de potentiels sur la quantité d&#x27;énergie annuelle, la puissance électrique de crête, la surfaces des capteurs, le potentiel d&#x27;autoconsommation de l&#x27;électricité, les émissions de GES évitées, les données économiques &#40;frais d&#x27;investissement, charges, recettes&#41;. Ces données sont calculées dans le cas d&#x27;une rentabilité économique optimale &#40;temps de retour sur investissement&#41; liée à l&#x27;autoconsommation du courant produit, tenant compte des tarifs d&#x27;achat et de revente de l&#x27;électricité en vigueur dans les différentes parties du Grand Genève. L&#x27;élaboration de cette couche s&#x27;inscrit dans le cadre du programme INTERREG V France-Suisse et du projet G2 Solaire. Ce dernier a permis à la fois de mettre à jour le précédent cadastre solaire genevois de 2016 &#40;sous mandat de l&#x27;OCEN - Office cantonal de l&#x27;énergie - et des SIG - Services industriels genevois&#41;, et de l&#x27;étendre sur le reste du Grand Genève incluant le canton de Genève, le District de Nyon et les communes francaises limitrophes des départements de l&#x27;Ain et de Haute Savoie. Le cadastre solaire a été calculé sur le Grand Genève en juin 2019 &#40;sur la base des données LIDAR 2017 Genève, 2014 France, 2015 Vaud&#41;, puis mis à jour en janvier 2022 uniquement sur le District de Nyon avec les données LIDAR de 2019. Les données calculées ne tiennent pas compte de bâtiments construits après ces dates.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/potentiel-photovoltaique-pv-par-batiment1)


## Metadata

- **identifier:** OCEN&#95;SOLAIRE&#95;PV&#95;BATIMENT@canton-geneve
- **publisher.name:** Office cantonal de l&#x27;énergie
- **organization.display&#95;name:** Kanton Genf
- **organization.url:** 
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** 
- **issued:** 2025-06-12 00:00:00
- **metadata&#95;created:** 2026-01-12T16:06:12.728582
- **metadata&#95;modified:** 2026-10-31T04:16:23.217806
- **Resource id:** 85e113dc-aa98-4ec8-a447-92b9c066c063
- **Resource name:** CSV
- **Resource package&#95;id:** e1d80b99-32b3-4f80-9185-e5560873f257
- **Resource description:** Service téléchargement extracteur
- **Resource issued:** 2025-06-12 00:00:00
- **Resource modified:** 2026-10-29 00:00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;open
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;open


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **CSV** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://extract.sitg.ge.ch/files/OCEN_SOLAIRE_PV_BATIMENT-csv.zip")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** Geraldine.Chollet@etat.ge.ch &#124; Office cantonal de l&#x27;énergie
